In [0]:
# Databricks Notebook
# ====================================================================
# CREATE GOLD LAYER VIEWS USING SPARK SQL
# ====================================================================

# ====================================================================
# Create Gold Schema
# ====================================================================

spark.sql("""
CREATE SCHEMA IF NOT EXISTS olist_lakehouse.gold
""")

# ====================================================================
# CREATE dim_customers
# ====================================================================

spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_customers AS

WITH customer_data AS (

    SELECT

        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        c.customer_zip_code_prefix,

        g.geolocation_lat,
        g.geolocation_lng,

        ROW_NUMBER() OVER (

            PARTITION BY c.customer_unique_id
            ORDER BY c.customer_zip_code_prefix

        ) AS row_num

    FROM olist_lakehouse.silver.olist_customers_dataset c

    LEFT JOIN
        olist_lakehouse.silver.olist_geolocation_dataset g
    ON
        c.customer_zip_code_prefix =
        g.geolocation_zip_code_prefix

)

SELECT

    ROW_NUMBER() OVER (
        ORDER BY customer_unique_id
    ) AS customer_key,

    customer_unique_id,

    customer_city,
    customer_state,
    customer_zip_code_prefix,

    geolocation_lat,
    geolocation_lng

FROM customer_data

WHERE row_num = 1

""")

print("SUCCESS : gold.dim_customers created")


# ====================================================================
# CREATE dim_products
# ====================================================================

spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_products AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY p.product_id
    ) AS product_key,

    p.product_id,

    CASE

        WHEN t.product_category_name IS NULL
             AND p.product_category_name = 'n/a'

        THEN p.product_category_name

        WHEN t.product_category_name IS NULL

        THEN CONCAT(
            p.product_category_name,
            ' (Translate)'
        )

        ELSE t.product_category_name_english

    END AS product_category,

    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,

    p.product_name_lenght,
    p.product_description_lenght,
    p.product_photos_qty

FROM olist_lakehouse.silver.olist_products_dataset p

LEFT JOIN
    olist_lakehouse.silver.product_category_name_translation t
ON
    p.product_category_name =
    t.product_category_name

""")

print("SUCCESS : gold.dim_products created")


# ====================================================================
# CREATE dim_sellers
# ====================================================================

spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_sellers AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY s.seller_id
    ) AS seller_key,

    s.seller_id,

    CASE

        WHEN g.geolocation_city IS NULL
        THEN s.seller_city

        ELSE g.geolocation_city

    END AS city,

    s.seller_state AS state,

    s.seller_zip_code_prefix AS zip_code,

    g.geolocation_lat AS latitude,
    g.geolocation_lng AS longitude

FROM olist_lakehouse.silver.olist_sellers_dataset s

LEFT JOIN
    olist_lakehouse.silver.olist_geolocation_dataset g
ON
    s.seller_zip_code_prefix =
    g.geolocation_zip_code_prefix

""")

print("SUCCESS : gold.dim_sellers created")


# ====================================================================
# CREATE dim_orders
# ====================================================================

spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_orders AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY o.order_id
    ) AS order_key,

    o.order_id,

    c.customer_key,

    o.order_status,

    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    o.dq_invalid_approval_timestamp_flag,
    o.dq_invalid_carrier_timestamp_flag,
    o.dq_invalid_customer_delivery_timestamp_flag,
    o.dq_invalid_estimated_delivery_timestamp_flag

FROM olist_lakehouse.silver.olist_orders_dataset o

LEFT JOIN
    olist_lakehouse.silver.olist_customers_dataset sc
ON
    o.customer_id = sc.customer_id

LEFT JOIN
    olist_lakehouse.gold.dim_customers c
ON
    sc.customer_unique_id = c.customer_unique_id

""")

print("SUCCESS : gold.dim_orders created")


# ====================================================================
# CREATE fact_order_items
# ====================================================================

spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.fact_order_items AS

SELECT

    d.order_key,

    p.product_key,
    s.seller_key,

    oi.price,

    oi.freight_value AS shipping_cost

FROM olist_lakehouse.silver.olist_order_items_dataset oi

LEFT JOIN
    olist_lakehouse.gold.dim_orders d
ON
    oi.order_id = d.order_id

LEFT JOIN
    olist_lakehouse.gold.dim_products p
ON
    oi.product_id = p.product_id

LEFT JOIN
    olist_lakehouse.gold.dim_sellers s
ON
    oi.seller_id = s.seller_id

""")

print("SUCCESS : gold.fact_order_items created")


# ====================================================================
# CREATE fact_payments
# ====================================================================

spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.fact_payments AS

SELECT

    d.order_key,

    p.payment_type,
    p.payment_installments,
    p.payment_value

FROM olist_lakehouse.silver.olist_order_payments_dataset p

LEFT JOIN
    olist_lakehouse.gold.dim_orders d
ON
    p.order_id = d.order_id

""")

print("SUCCESS : gold.fact_payments created")


# ====================================================================
# CREATE fact_reviews
# ====================================================================

spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.fact_reviews AS

SELECT

    d.order_key,

    r.review_score,
    r.review_creation_date

FROM olist_lakehouse.silver.olist_order_reviews_dataset r

LEFT JOIN
    olist_lakehouse.gold.dim_orders d
ON
    r.order_id = d.order_id

""")

print("SUCCESS : gold.fact_reviews created")


# ====================================================================
# VALIDATION
# ====================================================================

print("==================================================")
print("GOLD LAYER VIEWS CREATED SUCCESSFULLY")
print("==================================================")



In [0]:
display(
    spark.sql("""
        SHOW TABLES IN olist_lakehouse.gold
    """)
)